In [1]:
import pandas as pd
import numpy as np
import re

# Load raw datasets
df_roman = pd.read_csv('../data/raw/rufnd_roman_urdu.csv')
df_amjad = pd.read_excel('../data/raw/amjad_urdu.xlsx')
df_urdu = pd.read_csv('../data/raw/Urdu_News_Data.csv')

print("All datasets loaded")
print("RUFND:", df_roman.shape)
print("Amjad:", df_amjad.shape)
print("Urdu News:", df_urdu.shape)

All datasets loaded
RUFND: (3800, 3)
Amjad: (4097, 3)
Urdu News: (10444, 3)


In [2]:
# Standardize column names
df_roman = df_roman.rename(columns={
    'Roman News Items': 'text',
    'Label': 'label'
})[['text', 'label']]

df_amjad = df_amjad.rename(columns={
    'news': 'text',
    'label': 'label'
})[['text', 'label']]

df_urdu = df_urdu.rename(columns={
    'News Items': 'text',
    'Label': 'label'
})[['text', 'label']]

print("RUFND columns:", df_roman.columns.tolist())
print("Amjad columns:", df_amjad.columns.tolist())
print("Urdu News columns:", df_urdu.columns.tolist())

RUFND columns: ['text', 'label']
Amjad columns: ['text', 'label']
Urdu News columns: ['text', 'label']


In [3]:
# Standardize labels to lowercase fake/real
df_roman['label'] = df_roman['label'].str.lower().replace({'true': 'real'})
df_amjad['label'] = df_amjad['label'].str.lower()
df_urdu['label'] = df_urdu['label'].str.lower().replace({'true': 'real'})

# Verify
print("=== RUFND Labels ===")
print(df_roman['label'].value_counts())

print("\n=== Amjad Labels ===")
print(df_amjad['label'].value_counts())

print("\n=== Urdu News Labels ===")
print(df_urdu['label'].value_counts())

=== RUFND Labels ===
label
fake    2000
real    1800
Name: count, dtype: int64

=== Amjad Labels ===
label
fake     2455
real     1642
Name: count, dtype: int64

=== Urdu News Labels ===
label
real     6340
fake     4104
Name: count, dtype: int64


In [4]:
def clean_text(text):
    if pd.isnull(text):
        return np.nan
    
    # Remove leading/trailing whitespace
    text = str(text).strip()
    
    # Remove leading semicolons and quotes (Amjad dataset noise)
    text = re.sub(r'^[;"\s]+', '', text)
    
    # Remove newline characters
    text = re.sub(r'\n+', ' ', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

# Apply cleaning
df_roman['text'] = df_roman['text'].apply(clean_text)
df_amjad['text'] = df_amjad['text'].apply(clean_text)
df_urdu['text'] = df_urdu['text'].apply(clean_text)

# Verify on Amjad - check first row before and after
print("Sample cleaned Amjad text:")
print(df_amjad['text'][0])

print("\nSample cleaned Urdu News text:")
print(df_urdu['text'][0])

Sample cleaned Amjad text:
تفصیلات کے مطابق بحریہ ٹاؤن کی جانب سے بتیس قیدیوں کی رہائی کے لیے ارش، دمن اوردیت کی مد میں چونسٹھ لاکھ روپے کی رقم ادا کی گئی۔ یہ قیدی سزاپوری کر چکے تھے لیکن جرمانہ ادا نہیں کرسکے تھے۔ اس سے پہلے بھی مخیرحضرات کی جانب سے ادائیگی پر قیدیوں کو رہا کیا جاتا رہا ہے۔"; ; "تفصیلات کے مطابق اسلام پورہ میں بچوں سے چلنے والی گولی سے دادا جاں بحق ہوگئے، حادثے کی اطلاع ملنے پر پولیس جائے وقوعہ پر پہنچ گئی اور لاش قبضہ میں لے کر پوسٹمارٹم کیلئے مردہ خانے منتقل کردی۔"; پولیس نے واقعہ کی تفتیش شروع کرتے ہوئے اہلخانہ کے بیانات قلمبند کرلئے۔; "پولیس کا کہنا ہے کہ مقتولہ نصرت بی بی 3 بچوں کی ماں تھی، جائے وقوعہ سے شواہد اکٹھے کرلیے ہیں اور ملزم کو جلد گرفتار کرلیں گے۔"; تفصیلات کے مطابق برکی کے علاقے میں جائیداد کی خاطر بھائی نے بہن کو قتل کر دیا۔ حادثہ کی اطلاع ملنے پر پولیس جائے وقوعہ پر پہنچ گئی اور نعش قبضہ میں لے کر پوسٹمارٹم کیلئے مردہ خانے منتقل کر دی۔; "لاہورسمیت صوبہ بھرکے پولیس افسران کے پاس روزانہ ہزاروں شہری اپنی شکایات کے ازالے کے لیے دفاتر کے چکرلگاتے ہیں لی

In [5]:
# Check how many Amjad rows contain multiple concatenated stories
amjad_multirow = df_amjad['text'].str.contains(';"', na=False).sum()
print(f"Amjad rows with concatenated stories: {amjad_multirow}")
print(f"Total Amjad rows: {len(df_amjad)}")
print(f"Percentage affected: {round(amjad_multirow/len(df_amjad)*100, 2)}%")

# Show a clean row for comparison
print("\nExample of a clean Amjad row:")
clean_rows = df_amjad[~df_amjad['text'].str.contains(';"', na=False)]
print(clean_rows['text'].iloc[0])

Amjad rows with concatenated stories: 0
Total Amjad rows: 4097
Percentage affected: 0.0%

Example of a clean Amjad row:
تفصیلات کے مطابق بحریہ ٹاؤن کی جانب سے بتیس قیدیوں کی رہائی کے لیے ارش، دمن اوردیت کی مد میں چونسٹھ لاکھ روپے کی رقم ادا کی گئی۔ یہ قیدی سزاپوری کر چکے تھے لیکن جرمانہ ادا نہیں کرسکے تھے۔ اس سے پہلے بھی مخیرحضرات کی جانب سے ادائیگی پر قیدیوں کو رہا کیا جاتا رہا ہے۔"; ; "تفصیلات کے مطابق اسلام پورہ میں بچوں سے چلنے والی گولی سے دادا جاں بحق ہوگئے، حادثے کی اطلاع ملنے پر پولیس جائے وقوعہ پر پہنچ گئی اور لاش قبضہ میں لے کر پوسٹمارٹم کیلئے مردہ خانے منتقل کردی۔"; پولیس نے واقعہ کی تفتیش شروع کرتے ہوئے اہلخانہ کے بیانات قلمبند کرلئے۔; "پولیس کا کہنا ہے کہ مقتولہ نصرت بی بی 3 بچوں کی ماں تھی، جائے وقوعہ سے شواہد اکٹھے کرلیے ہیں اور ملزم کو جلد گرفتار کرلیں گے۔"; تفصیلات کے مطابق برکی کے علاقے میں جائیداد کی خاطر بھائی نے بہن کو قتل کر دیا۔ حادثہ کی اطلاع ملنے پر پولیس جائے وقوعہ پر پہنچ گئی اور نعش قبضہ میں لے کر پوسٹمارٹم کیلئے مردہ خانے منتقل کر دی۔; "لاہورسمیت صوبہ بھرک

In [6]:
# Drop Amjad dataset - structural data quality issue
# Multiple concatenated stories per row with single label
# Not suitable for single-document classification task

print("Proceeding with two datasets only:")
print(f"RUFND Roman Urdu: {df_roman.shape[0]} rows")
print(f"Urdu News: {df_urdu.shape[0]} rows")
print(f"Total: {df_roman.shape[0] + df_urdu.shape[0]} rows")

Proceeding with two datasets only:
RUFND Roman Urdu: 3800 rows
Urdu News: 10444 rows
Total: 14244 rows


In [7]:
# Drop 4 missing rows from Urdu News
df_urdu = df_urdu.dropna(subset=['text'])

# Remove empty strings
df_urdu = df_urdu[df_urdu['text'].str.strip() != '']
df_roman = df_roman[df_roman['text'].str.strip() != '']

# Reset indexes
df_urdu = df_urdu.reset_index(drop=True)
df_roman = df_roman.reset_index(drop=True)

print("After cleaning:")
print(f"RUFND Roman Urdu: {df_roman.shape[0]} rows")
print(f"Urdu News: {df_urdu.shape[0]} rows")
print(f"Total: {df_roman.shape[0] + df_urdu.shape[0]} rows")

After cleaning:
RUFND Roman Urdu: 3800 rows
Urdu News: 10440 rows
Total: 14240 rows


In [8]:
# Add source column
df_roman['source'] = 'rufnd_roman_urdu'
df_urdu['source'] = 'urdu_news'

# Add language column
df_roman['language'] = 'roman_urdu'
df_urdu['language'] = 'urdu_script'

# Verify
print("RUFND sample:")
print(df_roman.head(2))
print("\nUrdu News sample:")
print(df_urdu.head(2))

RUFND sample:
                                                text label            source  \
0  video sy pta chalta hai ky royal canadian moun...  fake  rufnd_roman_urdu   
1  Walmart cycles ko recycle krny ki bajaye $6.95...  fake  rufnd_roman_urdu   

     language  
0  roman_urdu  
1  roman_urdu  

Urdu News sample:
                                                text  label     source  \
0  سٹٹگارٹ (اُردو پوائنٹ اخبارتازہ ترین 03 دسمبر2...  fake   urdu_news   
1  کراچی ( آن لائن ) حکومت نے عوام کو باہرکریڈٹ ی...  fake   urdu_news   

      language  
0  urdu_script  
1  urdu_script  


In [9]:
# Save processed datasets separately
df_roman.to_csv('../data/processed/rufnd_processed.csv', index=False)
df_urdu.to_csv('../data/processed/urdu_news_processed.csv', index=False)

print("Processed datasets saved.")
print(f"RUFND: {df_roman.shape[0]} rows, {df_roman.shape[1]} columns")
print(f"Urdu News: {df_urdu.shape[0]} rows, {df_urdu.shape[1]} columns")
print(f"\nLabel distribution - RUFND:")
print(df_roman['label'].value_counts())
print(f"\nLabel distribution - Urdu News:")
print(df_urdu['label'].value_counts())

Processed datasets saved.
RUFND: 3800 rows, 4 columns
Urdu News: 10440 rows, 4 columns

Label distribution - RUFND:
label
fake    2000
real    1800
Name: count, dtype: int64

Label distribution - Urdu News:
label
real     6339
fake     4101
Name: count, dtype: int64
